## Try to build a model with pytorch use RNNs (the pictures are 28 * 28 and we should make it flatten and then build a model on the reshaped dataset)

In [2]:
from __future__ import print_function
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torchvision import datasets, transforms

In [8]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.LSTM(input_size =28, hidden_size=64, batch_first=True)
        self.batchnorm = nn.BatchNorm1d(64)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 10)

    def forward(self, _input):
        _input = _input.reshape(-1, 28, 28)
        output, hidden = self.rnn(_input)
        output = output[:, -1, :]
        output = self.batchnorm(output)
        output = self.dropout1(output)
        output = self.fc1(output)
        output = F.relu(output)
        output = self.dropout2(output)
        output = self.fc2(output)
        output = F.log_softmax(output, dim=1)
        return output

In [9]:
def train(model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 10 == 0:
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} ({100 * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}')

In [10]:
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
        
    test_loss /+ len(test_loader.dataset)

    print(f'\nTest set: Avatage loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({100 * correct / len(test_loader.dataset):.0f}%)\n')

In [11]:
torch.manual_seed(42)
use_cuda = torch.cuda.is_available()

device = torch.device('cuda' if use_cuda else 'cpu')

kwargs = {'num_workers': 1, 'pin_memory': True} if use_cuda else {}
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST(
        'data',
        train=True,
        download=True,
        transform=transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307), (0.3081))
        ])
    ),
    batch_size=1000,
    shuffle=True,
    **kwargs
)

test_loader = torch.utils.data.DataLoader(
    datasets.MNIST(
        'data',
        train=False,
        transform=transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307), (0.3081))
        ])
    ),
    batch_size=1000,
    shuffle=True,
    **kwargs
)

model = Model().to(device)
optimizer = optim.Adadelta(model.parameters(), lr=0.1)

scheduler = StepLR(optimizer=optimizer, step_size=1, gamma=0.7)

for epoch in range(1, 10):
    train(model, device, train_loader, optimizer, epoch)
    test(model, device, test_loader)
    scheduler.step()

torch.save(model.state_dict(), 'model/mnist_rnn.pt')

Train Epoch: 1 [0/60000 (0%)]	Loss: 2.424903
Train Epoch: 1 [10000/60000 (17%)]	Loss: 2.351499
Train Epoch: 1 [20000/60000 (33%)]	Loss: 2.277099
Train Epoch: 1 [30000/60000 (50%)]	Loss: 2.194043
Train Epoch: 1 [40000/60000 (67%)]	Loss: 2.151476
Train Epoch: 1 [50000/60000 (83%)]	Loss: 2.083858

Test set: Avatage loss: 20643.8955, Accuracy: 3607/10000 (36%)

Train Epoch: 2 [0/60000 (0%)]	Loss: 2.019774
Train Epoch: 2 [10000/60000 (17%)]	Loss: 2.006160
Train Epoch: 2 [20000/60000 (33%)]	Loss: 1.943033
Train Epoch: 2 [30000/60000 (50%)]	Loss: 1.916093
Train Epoch: 2 [40000/60000 (67%)]	Loss: 1.923010
Train Epoch: 2 [50000/60000 (83%)]	Loss: 1.834121

Test set: Avatage loss: 16900.6698, Accuracy: 4479/10000 (45%)

Train Epoch: 3 [0/60000 (0%)]	Loss: 1.863406
Train Epoch: 3 [10000/60000 (17%)]	Loss: 1.788552
Train Epoch: 3 [20000/60000 (33%)]	Loss: 1.782813
Train Epoch: 3 [30000/60000 (50%)]	Loss: 1.766866
Train Epoch: 3 [40000/60000 (67%)]	Loss: 1.736352
Train Epoch: 3 [50000/60000 (83%)]	